In [2]:
from pyspark.sql import SparkSession
import os
import json
from pyspark.sql.functions import col, when, length

In [3]:
OJDBC = "/home/ceci/jars/ojdbc17.jar"

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f'--jars "{OJDBC}" '
    f'--driver-class-path "{OJDBC}" '
    "pyspark-shell"
)

In [4]:
spark = SparkSession.builder \
    .appName("prova") \
    .master("local[1]") \
    .getOrCreate()

26/02/25 10:58:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
spark._jvm.java.lang.Class.forName("oracle.jdbc.OracleDriver")

JavaObject id=o32

In [4]:
sql = "select * from GRP02_RUO"
user = "VGLSA"
PW = "VGLSA"
idper = 200911

In [7]:
sql_1 = f"select * from GRP02_UNT PARTITION(P_{idper})"

df_1 = (spark.read \
    .format("jdbc") \
    .option("url","jdbc:oracle:thin:@//172.23.64.1:1521/ORCL") \
    .option("driver", "oracle.jdbc.OracleDriver") \
    .option("query", sql_1) \
    .option("user", user) \
    .option("password", PW) \
    .load()
    )
    #.option("fetchsize", 1000) # read 1000 records in a batch 
    #.option("numPartitions",5) # Query the JDBC table in Parallel

In [8]:
df_1.printSchema()

root
 |-- ID_PER: decimal(6,0) (nullable = true)
 |-- COD_UNT: decimal(5,0) (nullable = true)
 |-- DESC_UNT: string (nullable = true)



In [ ]:
df_1.write.jdbc

In [43]:
sql_2 = f"select * from COGE02_UNTACR PARTITION(P_{idper})" 

df_2 = (spark.read \
    .format("jdbc") \
    .option("url","jdbc:oracle:thin:@//172.23.64.1:1521/ORCL") \
    .option("driver", "oracle.jdbc.OracleDriver") \
    .option("query", sql_2) \
    .option("user", user) \
    .option("password", PW) \
    .load()
    )

In [25]:
df_2.printSchema()

root
 |-- ID_PER: decimal(6,0) (nullable = true)
 |-- COD_UNT: decimal(5,0) (nullable = true)
 |-- COD_ACR: string (nullable = true)



In [44]:
sql_3 = f"select * from ASL02_ACR PARTITION(P_{idper})"

df_3 = (spark.read \
    .format("jdbc") \
    .option("url","jdbc:oracle:thin:@//172.23.64.1:1521/ORCL") \
    .option("driver", "oracle.jdbc.OracleDriver") \
    .option("query", sql_3) \
    .option("user", user) \
    .option("password", PW) \
    .load()
    )

In [23]:
df_3.printSchema()

root
 |-- ID_PER: decimal(6,0) (nullable = true)
 |-- COD_ACR: string (nullable = true)
 |-- DESC_ACR: string (nullable = true)



In [50]:
df_join = df_1.join(df_2.select("COD_UNT", "COD_ACR"), on="COD_UNT")
df_join = df_join.join(df_3.select("COD_ACR", "DESC_ACR"), on="COD_ACR")

In [51]:
df_join.show()
df_join.printSchema()

+-------+-------+------+--------------------+--------------------+
|COD_ACR|COD_UNT|ID_PER|            DESC_UNT|            DESC_ACR|
+-------+-------+------+--------------------+--------------------+
|    RSA|  20310|200911|R.S.A.-I.S.F. SET...|             Anziani|
|    RSA|  15445|200911|    R.S.A. SAN LUIGI|             Anziani|
|    RSA|  40210|200911|     R.S.A. REGOLEDO|             Anziani|
|    IDR|  60320|200911|  INTRA SAN GIUSEPPE|Istituto di Riabi...|
|    CEM|  60324|200911|INTRA NUOVO SAN G...|Comunita' Educati...|
|    HOS|  60330|200911|INTRA SAN FRANCES...|             Ospizio|
|    RSD|  30340|200911|R.S.D.MONS.POGLIA...|Disabili Residenz...|
|    RSD|  20320|200911|R.S.D. SETTIMO MI...|Disabili Residenz...|
|    STD|  15660|200911|          SAN PIETRO|      Storici Diurni|
|    STD|  80300|200911|SANITARIA E ASSIS...|      Storici Diurni|
|    CDI|  15370|200911|C.D.NUOVO S.ELISA...|      Anziani Diurni|
|    CDI|  15360|200911|   C.D.ABBIATEGRASSO|      Anziani Diu

In [55]:
with open("../config/config_flussi_dm.json") as f:
    config = json.load(f)
config_corrente = config["flusso_prova"]

In [56]:
print(config_corrente["SCD_join_key"])
ax = config_corrente["SCD_columns"]
ax.append(config_corrente["SCD_join_key"])

CF


In [57]:
print(ax)

['SALARIO', 'CF']


In [48]:
type(ax.append(config_corrente["SCD_join_key"]))

NoneType

In [46]:
SCD1_columns = ax.append(config_corrente["SCD_join_key"]) + ["D_UPD", "D_INS", "D_INIVAL", "D_ENDVAL", "LAST_ID_PER"]
print(SCD1_columns)

TypeError: unsupported operand type(s) for +: 'NoneType' and 'list'

In [98]:
l = []
for a in x["02_tables"]:
    l.append(a)

In [101]:
sql = f"select {",".join(x["02_tables"][l[0]]["columns"])} from {l[0]} PARTITION(P_{idper})"

df_join = (spark.read \
    .format("jdbc") \
    .option("url","jdbc:oracle:thin:@//172.23.64.1:1521/ORCL") \
    .option("driver", "oracle.jdbc.OracleDriver") \
    .option("query", sql) \
    .option("user", user) \
    .option("password", PW) \
    .load()
    )

for t in l[1:]:
    sql_1 = f"select {",".join(x["02_tables"][t]["columns"])} from {t} PARTITION(P_{idper})"
    df_1 = (spark.read \
    .format("jdbc") \
    .option("url","jdbc:oracle:thin:@//172.23.64.1:1521/ORCL") \
    .option("driver", "oracle.jdbc.OracleDriver") \
    .option("query", sql_1) \
    .option("user", user) \
    .option("password", PW) \
    .load()
    )

    df_join = df_join.join(df_1, on=x["02_tables"][t]["join_key"])

In [102]:
df_join.show()
df_join.printSchema()
df_join.count()

+-------+-------+------+--------------------+--------------------+
|COD_ACR|COD_UNT|ID_PER|            DESC_UNT|            DESC_ACR|
+-------+-------+------+--------------------+--------------------+
|    RSA|  20310|200911|R.S.A.-I.S.F. SET...|             Anziani|
|    RSA|  15445|200911|    R.S.A. SAN LUIGI|             Anziani|
|    RSA|  40210|200911|     R.S.A. REGOLEDO|             Anziani|
|    IDR|  60320|200911|  INTRA SAN GIUSEPPE|Istituto di Riabi...|
|    CEM|  60324|200911|INTRA NUOVO SAN G...|Comunita' Educati...|
|    HOS|  60330|200911|INTRA SAN FRANCES...|             Ospizio|
|    RSD|  30340|200911|R.S.D.MONS.POGLIA...|Disabili Residenz...|
|    RSD|  20320|200911|R.S.D. SETTIMO MI...|Disabili Residenz...|
|    STD|  15660|200911|          SAN PIETRO|      Storici Diurni|
|    STD|  80300|200911|SANITARIA E ASSIS...|      Storici Diurni|
|    CDI|  15370|200911|C.D.NUOVO S.ELISA...|      Anziani Diurni|
|    CDI|  15360|200911|   C.D.ABBIATEGRASSO|      Anziani Diu

25

In [104]:
l = ["A", "B", "C"]

ol = [f"{x} AS OLD_{x}" for x in l]
st = f"select {",".join(ol)}"

print(st)

select A AS OLD_A,B AS OLD_B,C AS OLD_C


In [108]:
def equivalence(df, new_col, old_col):
        """
        Controlla se il vecchio valore di una colonna e uguale al nuovo
        """
        return df.withColumn(
            f"change_{new_col}",
            when(
               col(new_col) != col(old_col),
               True
            ).otherwise(False)
        )

In [5]:
df_test = spark.createDataFrame([('M', 27, 70, 171), 
                            ('F', 32, 50, None),
                            ('F', 24, 62, 164),
                            ('M', 25, 68, 169),
                            ('M', 50, 70, 172),
                            ('F', 46, 69, 174),
                            ('M', 36, 70, 173),
                            ('M', 34, 74, 180)],
                           ['gender', 'age', 'weigth','heigth'])

df_test_2 = spark.createDataFrame([('M', 27, 70, None), 
                            ('F', 32, 50, 165),
                            ('F', 24, 62, 164),
                            ('M', 25, 68, 169),
                            ('M', 50, 70, 172),
                            ('F', 46, 69, 174),
                            ('M', 36, 70, 173),
                            ('M', 34, 74, 180)],
                           ['gender', 'age', 'weigth','n_heigth'])

In [ ]:
df_test.filter(col("heigth").isNull()).count()

1

In [23]:
joined = df_test.join(df_test_2.select("age","n_heigth"), on="age") 

In [30]:
joined.show()

+---+------+------+------+--------+
|age|gender|weigth|heigth|n_heigth|
+---+------+------+------+--------+
| 24|     F|    62|   164|     164|
| 25|     M|    68|   169|     169|
| 27|     M|    70|   171|    NULL|
| 32|     F|    50|  NULL|     165|
| 34|     M|    74|   180|     180|
| 36|     M|    70|   173|     173|
| 46|     F|    69|   174|     174|
| 50|     M|    70|   172|     172|
+---+------+------+------+--------+



In [ ]:
joined.drop("ae").withColumnRenamed

+---+------+------+------+--------+
|age|gender|weigth|heigth|n_heigth|
+---+------+------+------+--------+
| 24|     F|    62|   164|     164|
| 25|     M|    68|   169|     169|
| 27|     M|    70|   171|    NULL|
| 32|     F|    50|  NULL|     165|
| 34|     M|    74|   180|     180|
| 36|     M|    70|   173|     173|
| 46|     F|    69|   174|     174|
| 50|     M|    70|   172|     172|
+---+------+------+------+--------+



In [122]:
joined = equivalence(joined, "heigth", "n_heigth")

In [123]:
joined.show()

+---+------+------+------+--------+-------------+
|age|gender|weigth|heigth|n_heigth|change_heigth|
+---+------+------+------+--------+-------------+
| 24|     F|    62|   164|     164|        false|
| 25|     M|    68|   169|     169|        false|
| 27|     M|    70|   171|     169|         true|
| 32|     F|    50|   165|     165|        false|
| 34|     M|    74|   180|     180|        false|
| 36|     M|    70|   173|     173|        false|
| 46|     F|    69|   174|     174|        false|
| 50|     M|    70|   172|     172|        false|
+---+------+------+------+--------+-------------+



In [25]:
def greater_than(df, col1, col2):
        """
        Controlla se la il valore in col1 e maggiore del valore in col2
        """
        return df.withColumn(
            f"greater_{col1}",
            when(
                col(col1) > col(col2),
                True
            ).otherwise(False)
        )

In [26]:
z = greater_than(joined, "heigth", "n_heigth")

In [39]:
xx = 'age'
z.select(z.getattr(xx)).show()

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `getattr` is not supported.

In [35]:
columns_oracle = ["a", "b", "c"]

print(f"""SET 
                {",".join([f"t.{x} = s.{x}" for x in columns_oracle])},
                LAST_ID_PER = s.ID_PER""")

SET 
                t.a = s.a,t.b = s.b,t.c = s.c,
                LAST_ID_PER = s.ID_PER


In [1]:
lp = [1,2,3]

In [2]:
print(lp)
print(*lp)

[1, 2, 3]
1 2 3


In [3]:
lp1 = []

if lp:
    print("yes lp!")

if lp1:
    print("yes lp1!")

yes lp!
